# Process Mining por Cluster (StudyChat) – sem simulação
Usa **sequências reais** por (usuário, tarefa) e os **rótulos de cluster** já calculados.

### Entradas esperadas
- `user_topic_cluster_labels_umap_order1.csv` (colunas: `userId,topic,cluster`)
- `per_user_topic_sequences.json` (lista de objetos com `userId`, `topic`, `sequences`)

### Saídas
- `outputs/cluster_<c>_heuristic_net.png` (se `pm4py` disponível)
- `outputs/cluster_<c>_avg_graph.png` (sempre)
- `outputs/cluster_metrics.csv` (sumário por cluster)
- `outputs/cluster_top_transitions.csv` (top transições por cluster)

Caso `pm4py` não esteja instalado no seu ambiente, o notebook segue gerando os grafos médios e as métricas.

In [1]:
!pip install pm4py
!pip install graphviz

In [2]:

import os
import json
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# Tentativa opcional de importar pm4py (se instalado no seu ambiente)
from pm4py.objects.log.obj import EventLog, Trace, Event
from pm4py.algo.discovery.heuristics import algorithm as heuristics_miner
from pm4py.visualization.heuristics_net import visualizer as hn_visualizer
PM4PY_AVAILABLE = True

os.makedirs("process_mining_outputs", exist_ok=True)
print("pm4py disponível?", PM4PY_AVAILABLE)


pm4py disponível? True


## 1) Carregar dados

In [3]:

labels_csv = "./da_artifacts/user_topic_cluster_labels_umap_order2.csv"
seq_json = "./da_artifacts/per_user_topic_sequences.json"

df_labels = pd.read_csv(labels_csv, dtype={"userId": str, "topic": str})
with open(seq_json, "r") as f:
    raw = json.load(f)

print("Exemplo labels_csv:")
display(df_labels.head())

print("Exemplo item do JSON:")
print(raw[0] if len(raw)>0 else "(vazio)")


Exemplo labels_csv:


,userId,topic,cluster
0,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,a5,3
1,81cbb5e0-9011-7077-7e4e-7aca1564666c,a6,3
2,e12ba500-f0e1-70f2-73ab-05c1cbc44b76,a3,0
3,f16b2520-6021-70ee-b980-7a50d804795d,a5,5
4,01eb0530-d011-70d4-70a6-3d8b36a0733f,a6,5


Exemplo item do JSON:
{'userId': '81fbe5c0-8001-7054-3ac3-3e9db3f6e198', 'topic': 'a5', 'sequences': [['Statement-non-opinion', 'Statement-non-opinion', 'Rhetorical-Question', 'Statement-non-opinion', 'Statement-non-opinion', 'Statement-non-opinion', 'Wh-Question', 'Statement-non-opinion', 'Statement-non-opinion', 'Statement-non-opinion', 'Statement-non-opinion', 'Statement-non-opinion', 'Statement-non-opinion', 'Statement-non-opinion', 'Statement-non-opinion', 'Statement-non-opinion', 'Other', 'Action-directive', 'Statement-non-opinion', 'Wh-Question', 'Statement-non-opinion', 'Statement-non-opinion'], ['Statement-non-opinion', 'Action-directive', 'Yes-No-Question', 'Yes-No-Question', 'Yes-No-Question', 'Statement-non-opinion', 'Statement-non-opinion', 'Statement-non-opinion', 'Wh-Question', 'Statement-non-opinion', 'Yes-No-Question', 'Yes-No-Question', 'Wh-Question', 'Statement-non-opinion', 'Statement-non-opinion', 'Statement-non-opinion', 'Yes-No-Question', 'Yes-No-Question', 'Stat

## 2) Expandir sequências em linhas de eventos

In [4]:

rows = []
for item in raw:
    user = item.get("userId")
    topic = item.get("topic")
    for seq_idx, seq in enumerate(item.get("sequences", [])):
        for turn_idx, act in enumerate(seq, start=1):
            rows.append({
                "userId": user,
                "topic": topic,
                "seq_index": seq_idx,
                "turn": turn_idx,
                "dialogue_act": act
            })

df_events = pd.DataFrame(rows)
print("Eventos expandidos:", df_events.shape)
display(df_events.head())


Eventos expandidos: (16364, 5)


,userId,topic,seq_index,turn,dialogue_act
0,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,a5,0,1,Statement-non-opinion
1,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,a5,0,2,Statement-non-opinion
2,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,a5,0,3,Rhetorical-Question
3,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,a5,0,4,Statement-non-opinion
4,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,a5,0,5,Statement-non-opinion


## 3) Associar (userId, topic) ao cluster

In [5]:

df = df_events.merge(df_labels, on=["userId", "topic"], how="inner")
df["cluster"] = df["cluster"].astype(str)
print("Eventos com cluster:", df.shape)
display(df.head())
print("Clusters:", sorted(df['cluster'].unique()))


Eventos com cluster: (16364, 6)


,userId,topic,seq_index,turn,dialogue_act,cluster
0,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,a5,0,1,Statement-non-opinion,3
1,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,a5,0,2,Statement-non-opinion,3
2,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,a5,0,3,Rhetorical-Question,3
3,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,a5,0,4,Statement-non-opinion,3
4,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,a5,0,5,Statement-non-opinion,3


Clusters: ['0', '1', '2', '3', '4', '5']


## 4) Funções auxiliares

In [19]:
DA_GROUPS = {
  "explicativo": [
    "Statement-non-opinion",
    "Statement-opinion",
    "Summarize/Reformulate",
    "Quotation",
    "Other"
  ],
  "perg. fato": [
    "Yes-No-Question",
    "Declarative Yes-No-Question",
    "Tag-Question"
  ],
  "perg. explo.": [
    "Wh-Question",
    "Declarative Wh-Question",
    "Open-Question",
    "Rhetorical-Question",
    "Or-Clause"
  ],
  "resp. aval": [
    "Yes Answers",
    "No Answers",
    "Other Answers",
    "Affirmative Non-yes Answers",
    "Negative Non-no Answers",
    "Response Acknowledgement",
    "Dispreferred Answers",
    "Maybe/Accept-part",
    "Downplayer"
  ],
  "reativo": [
    "Acknowledge (Backchannel)",
    "Backchannel in Question Form",
    "Collaborative Completion",
    "Repeat-phrase",
    "Hold Before Answer/Agreement",
    "Signal-non-understanding",
    "Hedge"
  ],
  "social": [
    "Appreciation",
    "Apology",
    "Thanking",
    "Conventional-opening",
    "Conventional-closing"
  ],
  "diretivo": [
    "Action-directive",
    "Offers, Options Commits"
  ],
  "metacomunicativo": [
    "Self-talk",
    "3rd-party-talk",
    "Uninterpretable"
  ]
}


def build_pm4py_log(df_cluster: pd.DataFrame):
    if not PM4PY_AVAILABLE:
        return None
    log = EventLog()
    for (user, topic, seq_idx), group in df_cluster.groupby(["userId", "topic", "seq_index"]):
        trace = Trace()
        group = group.sort_values("turn")
        for _, row in group.iterrows():
            trace.append(Event({
                "concept:name": row["dialogue_act"],
                "userId": row["userId"],
                "topic": row["topic"],
                "cluster": row["cluster"],
                "turn": int(row["turn"]),
                "seq_index": int(row["seq_index"]),
            }))
        log.append(trace)
    return log

def compute_transitions_from_events(df_cluster: pd.DataFrame):
    act_set = set()
    trans_counts = {}
    for _, g in df_cluster.groupby(["userId", "topic", "seq_index"]):
        g = g.sort_values("turn")
        acts = g["dialogue_act"].tolist()
        for a in acts:
            act_set.add(a)
        for i in range(len(acts)-1):
            pair = (acts[i], acts[i+1])
            trans_counts[pair] = trans_counts.get(pair, 0) + 1
    return trans_counts, sorted(act_set)

def avg_transition_matrix(trans_counts, acts):
    idx = {a:i for i,a in enumerate(acts)}
    M = np.zeros((len(acts), len(acts)), dtype=float)
    out_counts = np.zeros(len(acts), dtype=float)
    for (a, b), c in trans_counts.items():
        i, j = idx[a], idx[b]
        M[i, j] += c
        out_counts[i] += c
    for i in range(len(acts)):
        if out_counts[i] > 0:
            M[i, :] = M[i, :] / out_counts[i]
    return M, idx

def plot_avg_graph(M, acts, title, outpath, threshold=0.05):
    G = nx.DiGraph()
    for i, a1 in enumerate(acts):
        G.add_node(a1)
        for j, a2 in enumerate(acts):
            w = float(M[i, j])
            if w >= threshold:
                G.add_edge(a1, a2, weight=w)
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(10, 6))
    widths = [G[u][v]['weight'] * 5.0 for u, v in G.edges()]
    nx.draw_networkx_nodes(G, pos, node_size=1000)
    nx.draw_networkx_labels(G, pos, font_size=9)
    nx.draw_networkx_edges(G, pos, width=widths, arrows=True, arrowstyle='-|>')
    edge_labels = {(u, v): f"{G[u][v]['weight']:.2f}" for u, v in G.edges()}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

def entropy_rowwise(M):
    ent = []
    for row in M:
        p = row[row > 0]
        if len(p)==0:
            continue
        ent.append(-np.sum(p * np.log2(p)))
    return float(np.mean(ent)) if ent else 0.0

from collections import defaultdict

def build_act_to_group(groups):
    act_to_group = {}
    overlaps = defaultdict(list)
    for g, acts in groups.items():
        for a in acts:
            if a in act_to_group and act_to_group[a] != g:
                overlaps[a].extend([act_to_group[a], g])
            else:
                act_to_group[a] = g
    overlaps = {a: sorted(set(gs)) for a, gs in overlaps.items()}
    return act_to_group, overlaps

def group_freqs(act_freq, groups, include_unmapped=True, unmapped_key="_Unmapped"):
    act_to_group, overlaps = build_act_to_group(groups)
    total = int(act_freq.sum()) if act_freq is not None else 0
    counts = {g: 0 for g in groups.keys()}
    unmapped = 0
    for act, cnt in act_freq.items():
        g = act_to_group.get(act)
        if g is None:
            if include_unmapped:
                unmapped += int(cnt)
        else:
            counts[g] += int(cnt)
    if include_unmapped and unmapped > 0:
        counts[unmapped_key] = unmapped
    perc = {g: (counts[g] / total if total else 0.0) for g in counts}
    return counts, perc

def filter_and_sort_groups(perc_by_group: dict[str, float], threshold: float = 0.1) -> dict[str, float]:
    """
    Filtra e ordena grupos cujo percentual é maior que o threshold.
    Retorna um dicionário ordenado (do maior pro menor percentual).

    Args:
        perc_by_group: dicionário {grupo: percentual}
        threshold: valor mínimo de proporção (ex: 0.1 = 10%)
    """
    # Filtra os grupos acima do limiar
    filtered = {g: p for g, p in perc_by_group.items() if p > threshold}

    # Ordena por percentual decrescente
    sorted_groups = dict(sorted(filtered.items(), key=lambda kv: kv[1], reverse=True))

    return sorted_groups


def suggest_label(
    act_freq: pd.Series,
    ent: float,
    branching: float,
    ent_quantiles: tuple[float, float, float],
    br_quantiles: tuple[float, float, float],
) -> str:
    counts_by_group, perc_by_group = group_freqs(act_freq, DA_GROUPS)
    top_groups = filter_and_sort_groups(perc_by_group, threshold=0.1)

    q1_e, q2_e, q3_e = ent_quantiles
    q1_b, q2_b, q3_b = br_quantiles

    # níveis de entropia
    if ent < q1_e:
        ent_level = "low"
    elif ent > q3_e:
        ent_level = "high"
        # ent entre q1 e q3
    else:
        ent_level = "mid"

    # níveis de branching
    if branching < q1_b:
        br_level = "low"
    elif branching > q3_b:
        br_level = "high"
    else:
        br_level = "mid"

    # mapa completo (ent_level, br_level) -> rótulo
    label_map = {
        ("low", "low"): "Linear/diretivo",
        ("low", "mid"): "Ramificado com caminho preferencial",
        ("low", "high"): "Ramificado com baixa variação",
        ("mid", "low"): "Focado com variação local",
        ("mid", "mid"): "Misto/intermediário",
        ("mid", "high"): "Exploração estruturada",
        ("high", "low"): "Revisional/cíclico",
        ("high", "mid"): "Exploratório controlado",
        ("high", "high"): "Exploratório",
    }

    base = label_map[(ent_level, br_level)]
    return f"{base} + {list(top_groups.keys())}"




## 5) Processar por cluster

In [20]:

cluster_metrics = []
top_transitions_rows = []

ents = []
branchings = []
for c in sorted(df["cluster"].unique()):
    dfc = df[df["cluster"] == c].copy()
    trans_counts, acts = compute_transitions_from_events(dfc)
    M, idx = avg_transition_matrix(trans_counts, acts)
    branching_c = float(np.mean([(M[i,:] > 0).sum() for i in range(M.shape[0])]))
    ent_c = entropy_rowwise(M)
    ents.append(ent_c)
    branchings.append(branching_c)

ent_quantiles = tuple(np.quantile(ents, [0.25, 0.5, 0.75]))
br_quantiles  = tuple(np.quantile(branchings, [0.25, 0.5, 0.75]))


for c in sorted(df["cluster"].unique()):
    dfc = df[df["cluster"] == c].copy()
    print(f"=== Cluster {c} ===  eventos: {len(dfc)}  traces:", dfc.groupby(["userId","topic","seq_index"]).ngroups)
    
    log_c = build_pm4py_log(dfc)
    heu_net = heuristics_miner.apply_heu(log_c)
    gviz = hn_visualizer.apply(heu_net)
    out_png = os.path.join("process_mining_outputs", f"cluster_{c}_heuristic_net.png")
    hn_visualizer.save(gviz, out_png)
    print("Heuristic Net salvo em:", out_png)

    # Transições reais e grafo médio
    trans_counts, acts = compute_transitions_from_events(dfc)
    M, idx = avg_transition_matrix(trans_counts, acts)
    branching = float(np.mean([(M[i,:] > 0).sum() for i in range(M.shape[0])]))
    ent = entropy_rowwise(M)

    out_png2 = os.path.join("process_mining_outputs", f"cluster_{c}_avg_graph.png")
    plot_avg_graph(M, acts, f"Cluster {c} – Grafo de Transições Médias", out_png2, threshold=0.05)
    print("Grafo médio salvo em:", out_png2)

    act_freq = dfc["dialogue_act"].value_counts()
    mean_trace_len = dfc.groupby(["userId","topic","seq_index"])["turn"].max().mean()

    out_counts = {}
    for (a,b), cnt in trans_counts.items():
        out_counts[a] = out_counts.get(a, 0) + cnt
    top_pairs = sorted(trans_counts.items(), key=lambda kv: kv[1], reverse=True)[:25]
    for (a,b), cnt in top_pairs:
        prop = cnt / out_counts[a] if out_counts.get(a,0)>0 else 0.0
        top_transitions_rows.append({"cluster": c, "from": a, "to": b, "count": cnt, "prop_from": round(prop,4)})

    label = suggest_label(act_freq, ent, branching, ent_quantiles, br_quantiles)

    cluster_metrics.append({
        "cluster": c,
        "events": int(len(dfc)),
        "traces": int(dfc.groupby(["userId","topic","seq_index"]).ngroups),
        "unique_acts": int(len(acts)),
        "avg_branching": round(branching, 3),
        "avg_entropy": round(ent, 3),
        "mean_trace_length": round(float(mean_trace_len) if not math.isnan(mean_trace_len) else 0.0, 3),
        "suggested_label": label
    })

# Exportar
df_metrics = pd.DataFrame(cluster_metrics).sort_values("cluster")
df_metrics.to_csv(os.path.join("process_mining_outputs", "cluster_metrics.csv"), index=False)

df_top = pd.DataFrame(top_transitions_rows)
df_top.to_csv(os.path.join("process_mining_outputs", "cluster_top_transitions.csv"), index=False)

print("\nResumo por cluster:")
display(df_metrics)
print("\nTop transições por cluster (amostra):")
display(df_top.head(20))


=== Cluster 0 ===  eventos: 4005  traces: 340
Heuristic Net salvo em: process_mining_outputs/cluster_0_heuristic_net.png
Grafo médio salvo em: process_mining_outputs/cluster_0_avg_graph.png
=== Cluster 1 ===  eventos: 339  traces: 147
Heuristic Net salvo em: process_mining_outputs/cluster_1_heuristic_net.png
Grafo médio salvo em: process_mining_outputs/cluster_1_avg_graph.png
=== Cluster 2 ===  eventos: 261  traces: 57
Heuristic Net salvo em: process_mining_outputs/cluster_2_heuristic_net.png
Grafo médio salvo em: process_mining_outputs/cluster_2_avg_graph.png
=== Cluster 3 ===  eventos: 3492  traces: 382
Heuristic Net salvo em: process_mining_outputs/cluster_3_heuristic_net.png
Grafo médio salvo em: process_mining_outputs/cluster_3_avg_graph.png
=== Cluster 4 ===  eventos: 4369  traces: 417
Heuristic Net salvo em: process_mining_outputs/cluster_4_heuristic_net.png
Grafo médio salvo em: process_mining_outputs/cluster_4_avg_graph.png
=== Cluster 5 ===  eventos: 3898  traces: 384
Heurist

,cluster,events,traces,unique_acts,avg_branching,avg_entropy,mean_trace_length,suggested_label
0,0,4005,340,24,6.375,1.585,11.779,"Exploratório + ['explicativo', 'perg. explo.',..."
1,1,339,147,19,3.526,1.527,2.306,"Focado com variação local + ['explicativo', 'p..."
2,2,261,57,12,1.417,0.136,4.579,Linear/diretivo + ['explicativo']
3,3,3492,382,24,6.250,1.517,9.141,"Misto/intermediário + ['explicativo', 'perg. e..."
4,4,4369,417,26,6.462,1.535,10.477,"Exploratório + ['explicativo', 'perg. explo.',..."
5,5,3898,384,26,5.615,1.393,10.151,Ramificado com caminho preferencial + ['explic...



Top transições por cluster (amostra):


,cluster,from,to,count,prop_from
0,0,Statement-non-opinion,Statement-non-opinion,1435,0.6378
1,0,Statement-non-opinion,Action-directive,223,0.0991
2,0,Action-directive,Statement-non-opinion,220,0.6094
3,0,Statement-non-opinion,Yes-No-Question,146,0.0649
4,0,Wh-Question,Statement-non-opinion,127,0.5020
5,0,Yes-No-Question,Statement-non-opinion,124,0.5368
6,0,Statement-non-opinion,Wh-Question,120,0.0533
7,0,Rhetorical-Question,Statement-non-opinion,91,0.5871
8,0,Statement-non-opinion,Rhetorical-Question,84,0.0373
9,0,Statement-non-opinion,Quotation,73,0.0324
